# Phase 3: Risk Factor Scoring (Stage 1)

Compute each ergonomic risk factor per rider and bin to Low / Medium / High. Output: risk_profile.csv.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'model_ready.csv')
print(df.shape)

(182, 67)


## Step 1 — Inspect the columns we will score from

In [2]:
# Inputs feeding the 5 risk-factor formulas (Posture parked)
inputs = ['force_exertion', 'deliveries_num', 'work_hours_num',
          'vehicle_rank', 'vibration_index',
          'carrying_contact_rank', 'age_ord']
df[inputs].describe()

,force_exertion,deliveries_num,work_hours_num,vehicle_rank,vibration_index,carrying_contact_rank,age_ord
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,4.148352,25.494505,7.346154,1.483516,10.989011,1.945055,0.824176
std,2.645746,8.492927,1.965437,0.501107,4.969508,0.819144,0.861624
min,0.000000,5.000000,2.000000,1.000000,2.000000,1.000000,0.000000
25%,2.000000,15.000000,7.000000,1.000000,7.000000,1.000000,0.000000
50%,4.000000,25.000000,7.000000,1.000000,9.000000,2.000000,1.000000
75%,6.000000,35.000000,9.000000,2.000000,18.000000,2.000000,1.000000
max,10.000000,35.000000,9.000000,2.000000,18.000000,4.000000,3.000000


## Step 2 — Force risk (from Borg lifting item)

In [3]:
# Standard Borg CR10 action levels: 0-3 acceptable, 4-6 monitor, 7-10 intervene
labels = ['Low', 'Medium', 'High']
force_cat = pd.cut(df['force_exertion'], bins=[-0.5, 3, 6, 10], labels=labels)
df['force_risk']      = force_cat.astype(str)
df['force_risk_code'] = force_cat.cat.codes
df['force_risk'].value_counts()

force_risk
Low       90
Medium    57
High      35
Name: count, dtype: int64

## Step 3 — Repetition risk (deliveries per hour)

In [4]:
# Repetition rate: how many deliveries the rider does per hour worked
df['deliveries_per_hour'] = (df['deliveries_num'] / df['work_hours_num']).round(2)

# No published threshold for delivery work, so use sample terciles
rep_cat = pd.qcut(df['deliveries_per_hour'], q=3, labels=labels, duplicates='drop')
df['repetition_risk']      = rep_cat.astype(str)
df['repetition_risk_code'] = rep_cat.cat.codes

print('deliveries_per_hour stats:')
print(df['deliveries_per_hour'].describe().round(2))
df['repetition_risk'].value_counts()

deliveries_per_hour stats:
count    182.00
mean       3.59
std        1.14
min        0.56
25%        2.78
50%        3.75
75%        3.89
max        8.75
Name: deliveries_per_hour, dtype: float64


repetition_risk
Medium    84
Low       79
High      19
Name: count, dtype: int64

## Step 4 — Duration risk (continuous riding hours)

In [5]:
# Continuous riding/posture exposure: <=5 hrs Low, 6-7 hrs Medium, >7 hrs High
dur_cat = pd.cut(df['work_hours_num'], bins=[-1, 5, 7, 100], labels=labels)
df['duration_risk']      = dur_cat.astype(str)
df['duration_risk_code'] = dur_cat.cat.codes
df['duration_risk'].value_counts()

duration_risk
High      89
Medium    56
Low       37
Name: count, dtype: int64

## Step 5 — Vibration risk (vehicle x hours)

In [6]:
# vibration_index = vehicle_rank x work_hours_num was computed in Phase 2
# Sample terciles -> Low / Medium / High
vib_cat = pd.qcut(df['vibration_index'], q=3, labels=labels, duplicates='drop')
df['vibration_risk']      = vib_cat.astype(str)
df['vibration_risk_code'] = vib_cat.cat.codes

print('vibration_index distinct values:', sorted(df['vibration_index'].unique().tolist()))
df['vibration_risk'].value_counts()

vibration_index distinct values: [2, 4, 7, 8, 9, 14, 18]


vibration_risk
Medium    68
Low       67
High      47
Name: count, dtype: int64

## Step 6 — Contact Stress risk (carrying mode x hours x age factor)

In [7]:
# Contact stress combines how much the load contacts the body (carrying_contact_rank),
# how long (work_hours_num) and an age multiplier (older riders are more susceptible)
df['contact_stress_index'] = (df['carrying_contact_rank']
                              * df['work_hours_num']
                              * (1 + 0.1 * df['age_ord'])).round(2)

cs_cat = pd.qcut(df['contact_stress_index'], q=3, labels=labels, duplicates='drop')
df['contact_stress_risk']      = cs_cat.astype(str)
df['contact_stress_risk_code'] = cs_cat.cat.codes

print('contact_stress_index stats:')
print(df['contact_stress_index'].describe().round(2))
df['contact_stress_risk'].value_counts()

contact_stress_index stats:
count    182.00
mean      15.52
std        8.03
min        4.00
25%        9.00
50%       15.40
75%       19.80
max       39.60
Name: contact_stress_index, dtype: float64


contact_stress_risk
Low       68
Medium    58
High      56
Name: count, dtype: int64

## Step 7 — Posture risk (parked, placeholder)

In [8]:
# Posture per-rider linkage with the RULA/QEC xlsx is parked.
# Placeholder so the schema has all 6 factors.
df['posture_risk']      = 'Not Assessed'
df['posture_risk_code'] = np.nan
df['posture_risk'].value_counts()

posture_risk
Not Assessed    182
Name: count, dtype: int64

## Step 8 — Validate and save risk_profile.csv

In [9]:
# Every scored factor (5) must have all 182 labels filled
for c in ['force_risk', 'repetition_risk', 'duration_risk',
          'vibration_risk', 'contact_stress_risk']:
    assert df[c].notna().all(), c
    assert set(df[c].unique()) <= {'Low', 'Medium', 'High'}, c

risk_cols = ['force_risk', 'force_risk_code',
             'repetition_risk', 'repetition_risk_code',
             'duration_risk', 'duration_risk_code',
             'vibration_risk', 'vibration_risk_code',
             'contact_stress_risk', 'contact_stress_risk_code',
             'posture_risk', 'posture_risk_code']

profile = pd.DataFrame({'rider_id': df.index, 'timestamp': df['Timestamp']})
for c in risk_cols:
    profile[c] = df[c]

out = ROOT / 'data' / 'processed' / 'risk_profile.csv'
profile.to_csv(out, index=False)
print('saved', out, profile.shape)
profile.head()

saved f:\Ergonomic_Project\data\processed\risk_profile.csv (182, 14)


,rider_id,timestamp,force_risk,force_risk_code,repetition_risk,repetition_risk_code,duration_risk,duration_risk_code,vibration_risk,vibration_risk_code,contact_stress_risk,contact_stress_risk_code,posture_risk,posture_risk_code
0,0,3/3/2026 0:41:28,Low,0,Medium,1,Low,0,Low,0,Low,0,Not Assessed,NaN
1,1,3/3/2026 0:44:51,Low,0,Low,0,Medium,1,Low,0,High,2,Not Assessed,NaN
2,2,3/3/2026 0:47:05,Low,0,Medium,1,High,2,High,2,Medium,1,Not Assessed,NaN
3,3,3/3/2026 0:49:52,Low,0,Medium,1,Low,0,Low,0,Medium,1,Not Assessed,NaN
4,4,3/3/2026 0:52:25,Low,0,Low,0,Medium,1,Medium,1,High,2,Not Assessed,NaN
